# 04 Business layer: calibration, risk bands, expected loss and approval cutoff
Calibrated default probabilities, risk bands, expected loss and a profit-based approval cutoff. Calibration and the cutoff are chosen on the 2014 validation loans and evaluated on the 2015 test loans.

#### Load predictions and money columns

In [ ]:
import os, sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Markdown, display
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss
from sklearn.model_selection import StratifiedKFold
from sklearn.calibration import calibration_curve
sns.set_theme(style="whitegrid")

pv = pd.read_parquet("../data/processed/valid_predictions.parquet")
pt = pd.read_parquet("../data/processed/test_predictions.parquet")
biz = pd.read_parquet("../data/processed/business_data.parquet")

money = ["id", "funded_amnt", "total_pymnt", "total_rec_prncp", "total_rec_int",
         "total_rec_late_fee", "recoveries", "collection_recovery_fee"]
pv = pv.merge(biz[money], on="id", how="left", validate="one_to_one")
pt = pt.merge(biz[money], on="id", how="left", validate="one_to_one")
print("valid:", pv.shape, "| test:", pt.shape)
print("Missing money values after merge (valid, test):", int(pv[money].isna().sum().sum()), int(pt[money].isna().sum().sum()))

#### Define profit and loss, and estimate LGD

In [ ]:
parts = biz["total_rec_prncp"] + biz["total_rec_int"] + biz["total_rec_late_fee"] + biz["recoveries"]
diff = (biz["total_pymnt"] - parts).abs()
print("total_pymnt = principal + interest + late fees + recoveries (within $1):")
print("   all loans:", round((diff <= 1).mean() * 100, 2), "% | charged-off loans:", round((diff[biz["default"] == 1] <= 1).mean() * 100, 2), "%")
print("Charged-off loans with some recovery:", round((biz.loc[biz["default"] == 1, "recoveries"] > 0).mean() * 100, 1), "%")

for d in (pv, pt):
    d["profit"] = d["total_pymnt"] - d["funded_amnt"]
    d["principal_loss"] = np.where(
        d["default"] == 1,
        (d["funded_amnt"] - d["total_rec_prncp"] - (d["recoveries"] - d["collection_recovery_fee"])).clip(lower=0),
        0.0)

co = biz[(biz["split"].isin(["train", "valid"])) & (biz["default"] == 1)].copy()
co["loss"] = (co["funded_amnt"] - co["total_rec_prncp"] - (co["recoveries"] - co["collection_recovery_fee"])).clip(lower=0)
co["lgd"] = (co["loss"] / co["funded_amnt"]).clip(0, 1)
LGD = co["loss"].sum() / co["funded_amnt"].sum()
print(f"\nCharged-off loans used for LGD (2012-2014): {len(co):,}")
print(f"LGD, dollar-weighted: {LGD:.3f} | simple mean: {co['lgd'].mean():.3f} | median: {co['lgd'].median():.3f}")
print(f"Average realized profit per loan: valid ${pv['profit'].mean():,.0f}, test ${pt['profit'].mean():,.0f}")

#### Recalibrate probabilities on validation

In [ ]:
def logit(p):
    p = np.clip(np.asarray(p, dtype=float), 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))

def fit_cal(method, s, y):
    if method == "isotonic":
        return IsotonicRegression(out_of_bounds="clip", y_min=0, y_max=1).fit(np.asarray(s, dtype=float), y)
    return LogisticRegression(C=1e6, max_iter=1000).fit(logit(s).reshape(-1, 1), y)

def apply_cal(method, cal, s):
    if method == "isotonic":
        return cal.predict(np.asarray(s, dtype=float))
    return cal.predict_proba(logit(s).reshape(-1, 1))[:, 1]

CAL, cal_info = {}, {}
for fs in ["B", "A"]:
    sc = f"{fs}_LightGBM"
    s, y = pv[sc].values, pv["default"].values
    oof = {m: np.zeros(len(s)) for m in ["isotonic", "platt"]}
    for tr_i, te_i in StratifiedKFold(5, shuffle=True, random_state=42).split(s, y):
        for m in oof:
            oof[m][te_i] = apply_cal(m, fit_cal(m, s[tr_i], y[tr_i]), s[te_i])
    brier_cv = {m: brier_score_loss(y, oof[m]) for m in oof}
    method = min(brier_cv, key=brier_cv.get)
    cal = fit_cal(method, s, y)
    CAL[fs] = (method, cal)
    pv[f"pd_{fs}"] = apply_cal(method, cal, s)
    pt[f"pd_{fs}"] = apply_cal(method, cal, pt[sc].values)
    cal_info[fs] = {"method": method,
                    "brier_raw_test": brier_score_loss(pt["default"], pt[sc]),
                    "brier_cal_test": brier_score_loss(pt["default"], pt[f"pd_{fs}"]),
                    "mean_pd_test": float(pt[f"pd_{fs}"].mean()), "actual_test": float(pt["default"].mean())}
    print(f"Set {fs}: CV Brier on validation (isotonic {brier_cv['isotonic']:.5f}, platt {brier_cv['platt']:.5f}) -> {method} chosen")
    print(f"        test Brier raw {cal_info[fs]['brier_raw_test']:.5f} -> calibrated {cal_info[fs]['brier_cal_test']:.5f} | "
          f"mean PD {cal_info[fs]['mean_pd_test']:.4f} vs actual {cal_info[fs]['actual_test']:.4f}")

fig, ax = plt.subplots(figsize=(5.5, 5))
for col, label in [("B_LightGBM", "Set B raw score"), ("pd_B", "Set B recalibrated PD")]:
    fp, mp = calibration_curve(pt["default"], pt[col], n_bins=10, strategy="quantile")
    ax.plot(mp, fp, marker="o", label=label)
top = max(ax.get_xlim()[1], ax.get_ylim()[1])
ax.plot([0, top], [0, top], "k--", label="Perfect")
ax.set_xlabel("Average predicted probability")
ax.set_ylabel("Actual default rate (test)")
ax.set_title("Calibration before and after recalibration")
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig("../reports/figures/H0_calibration_before_after.png", dpi=150)
plt.show()
cb = cal_info["B"]
display(Markdown(f"**Takeaway:** a {cb['method']} recalibration fitted on the 2014 validation loans changes the test Brier score of the set B model from "
                 f"{cb['brier_raw_test']:.5f} to {cb['brier_cal_test']:.5f}. Average predicted default probability on 2015 is "
                 f"{cb['mean_pd_test']:.3f} versus an actual {cb['actual_test']:.3f}: recalibration cannot remove the drift because 2015 loans were riskier than 2014 loans."))

#### Risk bands vs LendingClub grades (set B is the primary model)

In [ ]:
SC, PDC = "B_LightGBM", "pd_B"
pt["band"] = pd.qcut(pt[SC].rank(method="first"), 5, labels=[1, 2, 3, 4, 5]).astype(int)
bt = pt.groupby("band").agg(loans=("id", "count"), avg_score=(SC, "mean"), avg_pd=(PDC, "mean"),
                            actual=("default", "mean"), avg_loan=("loan_amnt", "mean"))
bt_show = bt.copy()
bt_show["avg_pd"] *= 100
bt_show["actual"] *= 100
print("Risk bands on the 2015 test loans (band 1 = lowest predicted risk):")
print(bt_show.round(2).to_string())
gt = pt.groupby("grade").agg(loans=("id", "count"), actual=("default", "mean"))
print("\nLendingClub grades on the same loans:")
print((gt.assign(actual=gt["actual"] * 100)).round(2).to_string())
print("\nShare of each model band (rows) falling in each LendingClub grade (%):")
print(pd.crosstab(pt["band"], pt["grade"], normalize="index").mul(100).round(1).to_string())

mono = bool(bt["actual"].is_monotonic_increasing)
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
axes[0].bar(bt.index.astype(str), bt["actual"] * 100)
axes[0].set_title("Model risk bands (set B, LightGBM)")
axes[0].set_xlabel("Risk band (1 = safest)")
axes[0].set_ylabel("Actual default rate (%)")
axes[1].bar(gt.index.astype(str), gt["actual"] * 100, color="tab:orange")
axes[1].set_title("LendingClub grades")
axes[1].set_xlabel("Grade")
plt.tight_layout()
plt.savefig("../reports/figures/H1_risk_bands_vs_grades.png", dpi=150)
plt.show()
display(Markdown(f"**Takeaway:** actual default rates across the five model bands "
                 f"{'rise steadily' if mono else 'do not rise perfectly steadily'} from {bt['actual'].iloc[0] * 100:.1f}% (band 1) to "
                 f"{bt['actual'].iloc[-1] * 100:.1f}% (band 5), a {bt['actual'].iloc[-1] / bt['actual'].iloc[0]:.1f}x spread; "
                 f"LendingClub grades run from {gt['actual'].iloc[0] * 100:.1f}% (grade A) to {gt['actual'].max() * 100:.1f}% at their highest."))

#### Expected loss by band

In [ ]:
pt["el"] = pt[PDC] * LGD * pt["loan_amnt"]
g = pt.groupby("band").agg(loans=("id", "count"), avg_pd=(PDC, "mean"), total_loan_amnt=("loan_amnt", "sum"),
                           expected_loss=("el", "sum"), realized_loss=("principal_loss", "sum"), funded=("funded_amnt", "sum"))
g["expected_loss_rate_%"] = g["expected_loss"] / g["total_loan_amnt"] * 100
g["realized_loss_rate_%"] = g["realized_loss"] / g["funded"] * 100
el_tbl = pd.DataFrame({"loans": g["loans"], "avg_PD_%": g["avg_pd"] * 100, "LGD_used_%": LGD * 100,
                       "expected_loss_rate_%": g["expected_loss_rate_%"], "realized_loss_rate_%": g["realized_loss_rate_%"],
                       "expected_loss_$M": g["expected_loss"] / 1e6, "realized_loss_$M": g["realized_loss"] / 1e6})
print(el_tbl.round(2).to_string())
print(f"\nTotal expected loss ${g['expected_loss'].sum() / 1e6:,.1f}M vs realized principal loss ${g['realized_loss'].sum() / 1e6:,.1f}M")
el_tbl.round(3).to_csv("../reports/expected_loss_by_band.csv")

fig, ax = plt.subplots(figsize=(7, 4))
w = 0.38
x = np.arange(len(g))
ax.bar(x - w / 2, g["expected_loss_rate_%"], w, label="Expected (PD x LGD)")
ax.bar(x + w / 2, g["realized_loss_rate_%"], w, label="Realized principal loss")
ax.set_xticks(x)
ax.set_xticklabels(g.index.astype(str))
ax.set_xlabel("Risk band (1 = safest)")
ax.set_ylabel("Loss as % of loan amount")
ax.set_title("Expected vs realized loss rate by band (2015)")
ax.legend()
plt.tight_layout()
plt.savefig("../reports/figures/H2_expected_vs_realized_loss.png", dpi=150)
plt.show()
display(Markdown(f"**Takeaway:** expected loss rises from {g['expected_loss_rate_%'].iloc[0]:.1f}% of loan amount in band 1 to "
                 f"{g['expected_loss_rate_%'].iloc[-1]:.1f}% in band 5 (realized: {g['realized_loss_rate_%'].iloc[0]:.1f}% to "
                 f"{g['realized_loss_rate_%'].iloc[-1]:.1f}%). Total expected loss is ${g['expected_loss'].sum() / 1e6:,.1f}M against "
                 f"${g['realized_loss'].sum() / 1e6:,.1f}M realized; differences reflect the 2015 drift and the single average LGD used."))

#### Profit curves: cutoff chosen on validation

In [ ]:
def curve(df, sc, cutoffs):
    s = df[sc].values
    prof, dflt, fund = df["profit"].values, df["default"].values, df["funded_amnt"].values
    rows = []
    for c in cutoffs:
        m = s <= c
        n = m.sum()
        rows.append((c, m.mean() * 100, dflt[m].mean() * 100 if n else np.nan,
                     prof[m].sum(), prof[m].sum() / fund[m].sum() * 100 if n else np.nan))
    return pd.DataFrame(rows, columns=["cutoff", "approval_rate_%", "default_rate_%", "total_profit", "roi_%"])

def summarize(df, mask, name, base_profit=None):
    prof = df.loc[mask, "profit"].sum()
    fund = df.loc[mask, "funded_amnt"].sum()
    return {"strategy": name, "loans_approved": int(mask.sum()), "approval_rate_%": float(mask.mean() * 100),
            "default_rate_%": float(df.loc[mask, "default"].mean() * 100), "total_profit": float(prof),
            "roi_%": float(prof / fund * 100),
            "profit_vs_approve_all_%": float((prof - base_profit) / abs(base_profit) * 100) if base_profit else 0.0}

biz_results = {}
for fs in ["B", "A"]:
    sc = f"{fs}_LightGBM"
    cuts = np.unique(np.quantile(pv[sc], np.linspace(0.05, 1.0, 96)))
    vc = curve(pv, sc, cuts)
    best = float(vc.loc[vc["total_profit"].idxmax(), "cutoff"])
    tc_val = curve(pt, sc, cuts)
    tc_own = curve(pt, sc, np.unique(np.quantile(pt[sc], np.linspace(0.05, 1.0, 96))))
    hind = float(tc_own.loc[tc_own["total_profit"].idxmax(), "cutoff"])
    all_t = pd.Series(True, index=pt.index)
    base_t = pt["profit"].sum()
    rows = [summarize(pt, all_t, "Approve all", base_t),
            summarize(pt, pt["grade"].isin(["A", "B", "C"]), "Approve grades A-C only", base_t),
            summarize(pt, pt[sc] <= best, "Model cutoff (chosen on validation)", base_t),
            summarize(pt, pt[sc] <= hind, "Model cutoff (test hindsight, not usable)", base_t)]
    val_row = summarize(pv, pv[sc] <= best, "Model cutoff on validation", pv["profit"].sum())
    method, cal = CAL[fs]
    pd_at_cut = float(apply_cal(method, cal, np.array([best]))[0])
    biz_results[fs] = {"vcurve": vc, "tcurve": tc_val, "best": best, "pd_at_cutoff": pd_at_cut, "rows": rows, "val_row": val_row}

    print(f"===== Set {fs} ({sc}) =====")
    print(f"Cutoff chosen on VALIDATION: model score <= {best:.4f} (calibrated PD about {pd_at_cut * 100:.1f}%); "
          f"approves {val_row['approval_rate_%']:.1f}% of 2014 loans, profit {val_row['profit_vs_approve_all_%']:+.1f}% vs approve-all on validation.")
    show = pd.DataFrame(rows)
    show["total_profit"] = (show["total_profit"] / 1e6).round(2)
    show = show.rename(columns={"total_profit": "total_profit_$M"})
    print("Applied to the 2015 TEST loans:")
    print(show.round(2).to_string(index=False))
    print()

#### Profit curve chart

In [ ]:
r = biz_results["B"]
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
vc, bv = r["vcurve"], r["val_row"]
axes[0].plot(vc["approval_rate_%"], vc["total_profit"] / 1e6)
axes[0].scatter([bv["approval_rate_%"]], [bv["total_profit"] / 1e6], color="red", zorder=5, label="Chosen cutoff")
axes[0].set_title("Validation (2014): profit vs approval rate")
axes[0].set_xlabel("Approval rate (%)")
axes[0].set_ylabel("Total realized profit ($M)")
axes[0].legend()

tc = r["tcurve"]
axes[1].plot(tc["approval_rate_%"], tc["total_profit"] / 1e6, label="Model curve (validation cutoffs)")
named = {s["strategy"]: s for s in r["rows"]}
for name, color, label in [("Approve all", "black", "Approve all"),
                           ("Approve grades A-C only", "tab:orange", "Grades A-C only"),
                           ("Model cutoff (chosen on validation)", "red", "Chosen model cutoff")]:
    axes[1].scatter([named[name]["approval_rate_%"]], [named[name]["total_profit"] / 1e6], color=color, zorder=5, label=label)
axes[1].set_title("Test (2015): profit vs approval rate")
axes[1].set_xlabel("Approval rate (%)")
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.savefig("../reports/figures/H3_profit_curves.png", dpi=150)
plt.show()

sm, sa, sg = named["Model cutoff (chosen on validation)"], named["Approve all"], named["Approve grades A-C only"]
display(Markdown(
    f"**Takeaway:** the cutoff chosen on 2014 data approves {sm['approval_rate_%']:.1f}% of 2015 loans with a {sm['default_rate_%']:.1f}% default rate "
    f"(approve-all: {sa['default_rate_%']:.1f}%) and total profit of ${sm['total_profit'] / 1e6:,.2f}M versus ${sa['total_profit'] / 1e6:,.2f}M for "
    f"approve-all ({sm['profit_vs_approve_all_%']:+.1f}%) and ${sg['total_profit'] / 1e6:,.2f}M for grades A-C only ({sg['profit_vs_approve_all_%']:+.1f}%). "
    f"Return on funded capital: {sm['roi_%']:.2f}% vs {sa['roi_%']:.2f}% vs {sg['roi_%']:.2f}%."))

### Realized profit and return by LendingClub grade
Where does the money actually come from? Realized profit is total payments received minus the amount funded; return is profit divided by funded capital. This view needs no model.

In [ ]:
sys.path.append("../src")
from business import profit_by_group, fixed_volume_comparison

grade_profit = profit_by_group(pt, "grade")
print(grade_profit.round(2).to_string(index=False))
grade_profit.round(3).to_csv("../reports/profit_by_grade.csv", index=False)

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(grade_profit["grade"], grade_profit["roi_%"])
ax.axhline(0, color="black", linewidth=0.8)
for b, v in zip(bars, grade_profit["roi_%"]):
    ax.text(b.get_x() + b.get_width() / 2, v, f"{v:.1f}%", ha="center", va="bottom" if v >= 0 else "top", fontsize=9)
ax.set_xlabel("LendingClub grade")
ax.set_ylabel("Return on funded capital (%)")
ax.set_title("Realized return by LendingClub grade (2015 test)")
plt.tight_layout()
plt.savefig("../reports/figures/H6_profit_by_grade.png", dpi=150)
plt.show()

neg = grade_profit.loc[grade_profit["roi_%"] < 0, "grade"].tolist()
display(Markdown("**Takeaway:** " + (f"grades {', '.join(neg)} lost money in aggregate while every other grade earned a positive return; "
                                      if neg else "every grade earned a positive return; ")
                 + "interest income on risky loans offsets most default losses, which is why approving more loans keeps adding profit."))

### Realized profit and return by risk band
Where does the money actually come from? Realized profit is total payments received minus the amount funded; return is profit divided by funded capital.

In [ ]:
band_profit = profit_by_group(pt, "band")
print(band_profit.round(2).to_string(index=False))
band_profit.round(3).to_csv("../reports/profit_by_band.csv", index=False)

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(band_profit["band"].astype(str), band_profit["roi_%"])
ax.axhline(0, color="black", linewidth=0.8)
for b, v in zip(bars, band_profit["roi_%"]):
    ax.text(b.get_x() + b.get_width() / 2, v, f"{v:.1f}%", ha="center", va="bottom" if v >= 0 else "top", fontsize=9)
ax.set_xlabel("Risk band (1 = safest)")
ax.set_ylabel("Return on funded capital (%)")
ax.set_title("Realized return by model risk band (2015 test)")
plt.tight_layout()
plt.savefig("../reports/figures/H4_profit_by_band.png", dpi=150)
plt.show()

neg = band_profit.loc[band_profit["roi_%"] < 0, "band"].tolist()
first, last = band_profit.iloc[0], band_profit.iloc[-1]
msg = (f"**Takeaway:** realized return falls from {first['roi_%']:.1f}% in band 1 to {last['roi_%']:.1f}% in band 5. "
       + ("Every band earned a positive return, so approving more loans adds profit even in the riskiest band; this is why a profit-maximizing cutoff approves everyone."
          if not neg else f"Bands {', '.join(map(str, neg))} lost money in aggregate."))
display(Markdown(msg))

### Fixed-volume comparison: what is the ranking worth?
Total profit is maximized by approving every loan whose average return is positive, so the cutoff above approves everyone. A lender with limited capital cannot fund every loan. Here each ranking approves the same share of loans (the lowest-risk first) and realized profit is compared, with a random policy as the no-skill reference.

In [ ]:
rng = np.random.default_rng(42)
grade_score = pt["grade"].map({"A": 1, "B": 2, "C": 3, "D": 4, "E": 5, "F": 6, "G": 7}).astype(float) + rng.random(len(pt)) * 0.01  # random tie-break inside a grade
rankings = {
    "Model B (LightGBM)": pt["B_LightGBM"].values,
    "Model A (LightGBM, application only)": pt["A_LightGBM"].values,
    "LendingClub interest rate": pt["int_rate"].values + rng.random(len(pt)) * 1e-6,
    "LendingClub grade": grade_score.values,
}
rates = [0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
fv = fixed_volume_comparison(pt, rankings, rates, n_random=20, seed=42)
fv.round(3).to_csv("../reports/fixed_volume_comparison.csv", index=False)

print("Total profit ($M) at each approval rate:")
print((fv.pivot(index="approval_rate_%", columns="policy", values="total_profit") / 1e6).round(2).to_string())
print("\nReturn on funded capital (%):")
print(fv.pivot(index="approval_rate_%", columns="policy", values="roi_%").round(2).to_string())
print("\nDefault rate among approved loans (%):")
print(fv.pivot(index="approval_rate_%", columns="policy", values="default_rate_%").round(2).to_string())

fig, ax = plt.subplots(figsize=(8, 4.5))
for name, g in fv.groupby("policy"):
    ax.plot(g["approval_rate_%"], g["total_profit"] / 1e6, marker="o", label=name,
            linestyle="--" if name == "Random" else "-", color="gray" if name == "Random" else None)
ax.set_xlabel("Approval rate (%)")
ax.set_ylabel("Total realized profit ($M)")
ax.set_title("Profit at the same approval volume, by ranking method (2015 test)")
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig("../reports/figures/H5_fixed_volume_profit.png", dpi=150)
plt.show()

at = fv[fv["approval_rate_%"] == 70].set_index("policy")
usd = lambda v: f"-${abs(v):.1f}M" if v < 0 else f"${v:.1f}M"
p = lambda k: at.loc[k, "total_profit"] / 1e6
display(Markdown(
    f"**Takeaway (70% approval):** profit is {usd(p('Model B (LightGBM)'))} for the set B model, {usd(p('Model A (LightGBM, application only)'))} for the "
    f"application-only model, {usd(p('LendingClub interest rate'))} ranking by LendingClub's interest rate, {usd(p('LendingClub grade'))} by grade "
    f"and {usd(p('Random'))} for random selection. Set B minus random: {usd(p('Model B (LightGBM)') - p('Random'))}; "
    f"set B minus LendingClub's interest rate: {usd(p('Model B (LightGBM)') - p('LendingClub interest rate'))}."))

#### Save the business summary

In [ ]:
def to_plain(o):
    return o.item() if hasattr(o, "item") else str(o)

summary = {
    "n_loans": int(len(biz)),
    "LGD": float(LGD), "lgd_mean": float(co["lgd"].mean()), "lgd_median": float(co["lgd"].median()),
    "calibration": cal_info,
    "bands": bt.reset_index()[["band", "loans", "avg_pd", "actual"]].to_dict("records"),
    "grades": gt.reset_index()[["grade", "loans", "actual"]].to_dict("records"),
    "cutoff": {fs: {"score_cutoff": biz_results[fs]["best"], "pd_at_cutoff": biz_results[fs]["pd_at_cutoff"],
                    "val_approval_rate": biz_results[fs]["val_row"]["approval_rate_%"]} for fs in ["A", "B"]},
    "strategies": {fs: biz_results[fs]["rows"] for fs in ["A", "B"]},
}
with open("../reports/business_summary.json", "w") as f:
    json.dump(summary, f, indent=2, default=to_plain)
pd.DataFrame(biz_results["B"]["rows"]).round(3).to_csv("../reports/profit_strategies_test_setB.csv", index=False)
pd.DataFrame(biz_results["A"]["rows"]).round(3).to_csv("../reports/profit_strategies_test_setA.csv", index=False)
print("Saved reports/business_summary.json and the profit strategy tables.")

### Business layer summary
Model scores were recalibrated on the 2014 validation loans (Platt or isotonic, chosen by cross-validated Brier score). Test loans were sorted into five risk bands by score and their actual default rates compared with LendingClub grades. Expected loss = PD x LGD x loan amount, with LGD estimated from charged-off 2012-2014 loans (funded amount minus principal repaid minus net recoveries, dollar-weighted) using post-issue columns for this analysis only. Realized profit per loan = total_pymnt (which includes recoveries) minus funded_amnt.

**Approval cutoff.** The cutoff that maximizes validation profit approves 100% of loans, i.e. approve-all was profit-maximizing on 2014 loans. On the 2015 test loans the best hindsight cutoff approves 99% of loans for +0.3% profit, and approving only grades A-C lowers profit by 9.3%. On 2015 loans grades A to E all earned a positive return and only grades F and G (1,606 loans) lost money; on 2014 loans every grade earned a positive return. Because total profit is maximized by funding every loan with a positive average return, the value of the model is judged by realized profit and return by grade and by risk band, and by a fixed-volume comparison of ranking methods for a lender that cannot fund every loan.

Limitations: only approved loans are observed (selection bias), a single 2012-2015 vintage window, no macro-economic factors, and simplified profit (no servicing fees, funding costs, time value of money or collection costs).